# External Lab Evaluation with SapBERT Contrastive

This notebook builds an efficient deduplicated FAISS corpus from the full reference synonym CSV, embeds each unique normalized surface form once with the fine-tuned `sapbert_contrastive` model, and evaluates the externally labeled lab names while filtering identical normalized surface-form hits.

Outputs are saved under `results/sapbert_contrastive/external_lab_unique_surface/`.

## Install Dependencies

In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'torch>=2.1',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

# Try GPU FAISS first. If that wheel is unavailable in your Colab image, fall back to CPU FAISS.
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])


## Mount Drive and Locate Repo

In [ ]:
from pathlib import Path
import os
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer. Update the candidates in find_repo_dir().')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration

In [ ]:
import hashlib
import json
import time
from typing import Iterable

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from chem_synonymizer.embed import choose_device, encode_texts_to_memmap, load_hf_model, pool_outputs
from chem_synonymizer.index import build_faiss_index_from_memmap, faiss_gpu_available
from chem_synonymizer.evaluate import search_faiss_neighbors, pretty_metrics
from chem_synonymizer.utils import save_json, set_seed

DATA_DIR = REPO_DIR / 'data'
REFERENCE_CSV_PATH = DATA_DIR / 'CID-Synonym-full.csv'
LAB_LABELS_PATH = REPO_DIR / 'results' / 'lab_cid_mapping' / 'lab_labeled_for_evaluation.csv'
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'

RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_unique_surface'
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'
RESULTS_PLOTS_DIR = RESULTS_RUN_DIR / 'plots'
RESULTS_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def pick_scratch_root() -> Path:
    candidates = [
        Path('/content/local-scratch'),
        Path('/mnt/local-scratch'),
        Path('/local-scratch'),
        Path('/mnt/disks/local-scratch'),
        Path('/mnt/disks/local-ssd'),
        Path('/mnt/localssd'),
        Path('/content'),
    ]
    writable = [p for p in candidates if p.exists() and os.access(p, os.W_OK)]
    if not writable:
        return Path('/content')
    return max(writable, key=lambda p: shutil.disk_usage(p).free)


SCRATCH_ROOT = pick_scratch_root()
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / 'sapbert_contrastive_external_lab_unique_surface'
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'
WORK_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
LEGACY_WORK_ARTIFACTS_DIRS = [
    Path('/content/chem_synonymizer/sapbert_contrastive_external_lab_unique_surface/artifacts'),
]

CID_COL = 'cid'
NAME_COL = 'name'
SEED = 0
CPU_THREADS = max(1, os.cpu_count() or 8)
REFERENCE_CHUNKSIZE = 5_000_000
MAX_LENGTH = 96
CORPUS_BATCH_SIZE = 8192
QUERY_BATCH_SIZE = 16384
EMBED_DTYPE = 'float16'
POOLING = 'cls'
L2_NORMALIZE = True

# Full IVF-flat is too large for the 76M-surface external corpus on an 80 GB GPU.
# IVFPQ keeps the full candidate corpus but compresses the FAISS index enough to finish.
INDEX_TYPE = 'ivfpq'
NLIST = 16384
NPROBE = 512
TRAIN_SIZE = 3_000_000
ADD_BATCH_SIZE = 250_000
SEARCH_BATCH_SIZE = 8192
USE_FLOAT16_FAISS = True
USE_GPU_FAISS_IF_FITS = True
GPU_INDEX_MEMORY_LIMIT_GB = 70.0

K_VALUES = [1, 5, 10]
K_RETRIEVE = 256
RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS = True
RERANK_BATCH_SIZE = 512
EVALUATE_ONLY_QUERIES_WITH_NONEXACT_RELEVANT = True

REBUILD_PREPARED_CORPUS = False
REENCODE_CORPUS = False
REBUILD_FAISS_INDEX = False
REENCODE_QUERIES = False
RESTORE_LARGE_ARTIFACTS_FROM_RESULTS = True
PERSIST_LARGE_ARTIFACTS_TO_DRIVE = True
DELETE_LOCAL_LARGE_ARTIFACTS_AFTER_DRIVE_BACKUP = True
COPY_LARGE_ARTIFACTS_TO_RESULTS = PERSIST_LARGE_ARTIFACTS_TO_DRIVE
CHUNKED_MOVE_ARTIFACT_NAMES = {'corpus_embeddings.npy', 'lab_query_embeddings.npy'}
CHUNKED_MOVE_CHUNK_GB = 2
CHUNKED_MOVE_BUFFER_MB = 64
CHUNKED_MOVE_WRITE_CHECKSUMS = True
VERIFY_DRIVE_CHUNK_CHECKSUMS = False  # True performs a full checksum audit, but rereads every Drive chunk.
DELETE_PREPARE_DB_AFTER_PARQUET = True
ALLOW_CPU_IVF_FLAT_FALLBACK = False
M_PQ = 96
NBITS = 8

PREPARE_DB_PATH = WORK_ARTIFACTS_DIR / 'reference_prepare.duckdb'
CORPUS_PARQUET_PATH = WORK_ARTIFACTS_DIR / 'unique_surface_corpus.parquet'
CORPUS_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'corpus_embeddings.npy'
QUERY_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'lab_query_embeddings.npy'
FAISS_INDEX_PATH = WORK_ARTIFACTS_DIR / f'faiss_{INDEX_TYPE}.index'
NEIGHBOR_INDICES_PATH = WORK_ARTIFACTS_DIR / f'lab_neighbor_indices_k{K_RETRIEVE}_rerank{int(RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS)}.npy'
NEIGHBOR_CONFIG_PATH = NEIGHBOR_INDICES_PATH.with_suffix(NEIGHBOR_INDICES_PATH.suffix + '.config.json')

for required in [REFERENCE_CSV_PATH, LAB_LABELS_PATH, MODEL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)

set_seed(SEED)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Lab labels:', LAB_LABELS_PATH)
print('Model:', MODEL_DIR)
print('Scratch root:', SCRATCH_ROOT, shutil.disk_usage(SCRATCH_ROOT))
print('Work artifacts:', WORK_ARTIFACTS_DIR)
print('CPU threads:', CPU_THREADS)
print('Saved results:', RESULTS_RUN_DIR)


## Helpers

In [ ]:
def normalize_series(values: pd.Series) -> pd.Series:
    return (
        values.fillna('')
        .astype('string')
        .str.normalize('NFKC')
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def path_for_results(work_path: Path) -> Path:
    return RESULTS_ARTIFACTS_DIR / work_path.name


def parts_dir_for_results(work_path: Path) -> Path:
    return RESULTS_ARTIFACTS_DIR / f'{work_path.name}.parts'


def should_use_chunked_move(work_path: Path) -> bool:
    return work_path.name in CHUNKED_MOVE_ARTIFACT_NAMES


def artifact_storage_path(work_path: Path) -> Path:
    if should_use_chunked_move(work_path):
        return parts_dir_for_results(work_path)
    return path_for_results(work_path) if PERSIST_LARGE_ARTIFACTS_TO_DRIVE else work_path


def sidecar_suffixes() -> list[str]:
    return ['.meta.json', '.timing.json']


def copy_sidecars_to_dir(work_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    for suffix in sidecar_suffixes():
        sidecar = work_path.with_suffix(work_path.suffix + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, dest_dir / sidecar.name)


def restore_sidecars_from_dir(work_path: Path, src_dir: Path) -> None:
    for suffix in sidecar_suffixes():
        sidecar_src = src_dir / f'{work_path.name}{suffix}'
        if sidecar_src.exists():
            shutil.copy2(sidecar_src, work_path.with_suffix(work_path.suffix + suffix))


def chunk_manifest_path(work_path: Path) -> Path:
    return parts_dir_for_results(work_path) / 'manifest.json'


def load_chunk_manifest(work_path: Path) -> dict | None:
    manifest_path = chunk_manifest_path(work_path)
    if not manifest_path.exists():
        return None
    with manifest_path.open('r', encoding='utf-8') as f:
        return json.load(f)


def expected_chunk_size(manifest: dict, part_i: int) -> int:
    start = int(part_i) * int(manifest['chunk_size'])
    end = min(start + int(manifest['chunk_size']), int(manifest['original_size']))
    return int(end - start)


def file_fingerprint(path: Path) -> dict:
    if not path.exists():
        return {'path': str(path), 'exists': False}
    stat = path.stat()
    return {'path': str(path), 'size': int(stat.st_size), 'mtime_ns': int(stat.st_mtime_ns)}


def stable_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha1(raw).hexdigest()[:16]


def artifact_fingerprint(work_path: Path) -> dict:
    if should_use_chunked_move(work_path):
        manifest = load_chunk_manifest(work_path)
        if manifest is not None:
            chunks = manifest.get('chunks', {})
            return {
                'path': str(parts_dir_for_results(work_path)),
                'chunked': True,
                'original_size': int(manifest.get('original_size', 0)),
                'chunk_size': int(manifest.get('chunk_size', 0)),
                'n_chunks': int(manifest.get('n_chunks', 0)),
                'completed': bool(manifest.get('completed', False)),
                'chunks_hash': stable_hash(chunks),
            }
    result_path = path_for_results(work_path)
    if result_path.exists():
        return file_fingerprint(result_path)
    return file_fingerprint(work_path)


def chunk_sha256(path: Path, buffer_size: int | None = None) -> str:
    h = hashlib.sha256()
    buffer_size = buffer_size or int(CHUNKED_MOVE_BUFFER_MB * 1024 ** 2)
    with path.open('rb') as f:
        while True:
            block = f.read(buffer_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def manifest_chunks(manifest: dict) -> dict:
    chunks = manifest.setdefault('chunks', {})
    return chunks


def chunk_is_verified(part_path: Path, expected_size: int, expected_sha256: str | None = None, verify_checksum: bool = False) -> bool:
    if not part_path.exists() or part_path.stat().st_size != expected_size:
        return False
    if verify_checksum and expected_sha256:
        return chunk_sha256(part_path) == expected_sha256
    return True


def non_destructive_chunked_copy_to_results(work_path: Path) -> None:
    if not PERSIST_LARGE_ARTIFACTS_TO_DRIVE or not work_path.exists():
        return
    parts_dir = parts_dir_for_results(work_path)
    parts_dir.mkdir(parents=True, exist_ok=True)
    single_path = path_for_results(work_path)
    if single_path.exists():
        print(f'Deleting single-file Drive copy before chunked copy: {single_path}')
        single_path.unlink()

    chunk_size = int(CHUNKED_MOVE_CHUNK_GB * 1024 ** 3)
    buffer_size = int(CHUNKED_MOVE_BUFFER_MB * 1024 ** 2)
    original_size = int(work_path.stat().st_size)
    n_chunks = int((original_size + chunk_size - 1) // chunk_size)
    manifest = load_chunk_manifest(work_path) or {
        'file_name': work_path.name,
        'original_size': original_size,
        'chunk_size': chunk_size,
        'n_chunks': n_chunks,
        'mode': 'chunked_copy_or_move',
        'completed': False,
        'chunks': {},
    }
    if int(manifest.get('original_size', -1)) != original_size:
        raise RuntimeError(f'Existing chunk manifest size does not match local file for {work_path.name}.')
    chunks = manifest_chunks(manifest)
    print(f'Chunked copy for {work_path.name}: {original_size / 1024 ** 3:.2f} GB across {n_chunks} parts; local copy retained')

    with work_path.open('rb') as f:
        for part_i in range(n_chunks):
            start = part_i * chunk_size
            expected_size = expected_chunk_size(manifest, part_i)
            part_key = f'{part_i:05d}'
            part_path = parts_dir / f'{work_path.name}.part{part_key}'
            expected_sha = chunks.get(part_key, {}).get('sha256')
            if chunk_is_verified(part_path, expected_size, expected_sha, verify_checksum=VERIFY_DRIVE_CHUNK_CHECKSUMS):
                print(f'SKIP verified part {part_i + 1}/{n_chunks}')
                continue
            if part_path.exists():
                print(f'Deleting incomplete Drive part: {part_path.name}')
                part_path.unlink()
            print(f'COPY part {part_i + 1}/{n_chunks}: {expected_size / 1024 ** 3:.2f} GB')
            t0 = time.perf_counter()
            h = hashlib.sha256()
            f.seek(start)
            remaining = expected_size
            with part_path.open('wb') as out:
                while remaining > 0:
                    block = f.read(min(buffer_size, remaining))
                    if not block:
                        break
                    out.write(block)
                    h.update(block)
                    remaining -= len(block)
            if part_path.stat().st_size != expected_size:
                raise OSError(f'Chunk size mismatch for {part_path.name}.')
            chunks[part_key] = {'size': int(expected_size), 'sha256': h.hexdigest() if CHUNKED_MOVE_WRITE_CHECKSUMS else None}
            with chunk_manifest_path(work_path).open('w', encoding='utf-8') as f_manifest:
                json.dump(manifest, f_manifest, indent=2)
            print(f'DONE part {part_i + 1}/{n_chunks}; elapsed={(time.perf_counter() - t0) / 60:.1f} min')
    manifest['completed'] = True
    with chunk_manifest_path(work_path).open('w', encoding='utf-8') as f_manifest:
        json.dump(manifest, f_manifest, indent=2)
    copy_sidecars_to_dir(work_path, parts_dir)
    print(f'Chunked copy complete for {work_path.name}: {parts_dir}')


def destructive_chunked_move_to_results(work_path: Path) -> None:
    if not PERSIST_LARGE_ARTIFACTS_TO_DRIVE or not work_path.exists():
        return
    parts_dir = parts_dir_for_results(work_path)
    parts_dir.mkdir(parents=True, exist_ok=True)
    single_path = path_for_results(work_path)
    if single_path.exists():
        print(f'Deleting single-file Drive copy before chunked move: {single_path}')
        single_path.unlink()

    chunk_size = int(CHUNKED_MOVE_CHUNK_GB * 1024 ** 3)
    buffer_size = int(CHUNKED_MOVE_BUFFER_MB * 1024 ** 2)
    manifest = load_chunk_manifest(work_path)
    if manifest is None:
        original_size = int(work_path.stat().st_size)
        manifest = {
            'file_name': work_path.name,
            'original_size': original_size,
            'chunk_size': chunk_size,
            'n_chunks': int((original_size + chunk_size - 1) // chunk_size),
            'mode': 'reverse_destructive_move',
            'completed': False,
            'chunks': {},
        }
        with chunk_manifest_path(work_path).open('w', encoding='utf-8') as f:
            json.dump(manifest, f, indent=2)

    original_size = int(manifest['original_size'])
    n_chunks = int(manifest['n_chunks'])
    chunk_size = int(manifest['chunk_size'])
    chunks = manifest_chunks(manifest)
    print(f'Chunked destructive move for {work_path.name}: {original_size / 1024 ** 3:.2f} GB across {n_chunks} parts')

    with work_path.open('r+b') as f:
        for part_i in range(n_chunks - 1, -1, -1):
            start = part_i * chunk_size
            end = min(start + chunk_size, original_size)
            expected_size = end - start
            part_key = f'{part_i:05d}'
            part_path = parts_dir / f'{work_path.name}.part{part_key}'
            current_size = int(work_path.stat().st_size)
            expected_sha = chunks.get(part_key, {}).get('sha256')

            if chunk_is_verified(part_path, expected_size, expected_sha, verify_checksum=VERIFY_DRIVE_CHUNK_CHECKSUMS):
                if current_size > start:
                    f.truncate(start)
                print(f'SKIP verified part {part_i + 1}/{n_chunks}; local remaining={start / 1024 ** 3:.2f} GB')
                continue
            if current_size < end:
                raise RuntimeError(f'Missing Drive part {part_path.name}, but local file was already truncated below that range.')
            if part_path.exists():
                print(f'Deleting incomplete Drive part: {part_path.name}')
                part_path.unlink()

            print(f'COPY part {part_i + 1}/{n_chunks}: {expected_size / 1024 ** 3:.2f} GB')
            t0 = time.perf_counter()
            h = hashlib.sha256()
            f.seek(start)
            remaining = expected_size
            with part_path.open('wb') as out:
                while remaining > 0:
                    block = f.read(min(buffer_size, remaining))
                    if not block:
                        break
                    out.write(block)
                    h.update(block)
                    remaining -= len(block)
            if part_path.stat().st_size != expected_size:
                raise OSError(f'Chunk size mismatch for {part_path.name}; local file was not truncated.')
            chunks[part_key] = {'size': int(expected_size), 'sha256': h.hexdigest() if CHUNKED_MOVE_WRITE_CHECKSUMS else None}
            with chunk_manifest_path(work_path).open('w', encoding='utf-8') as f_manifest:
                json.dump(manifest, f_manifest, indent=2)
            f.truncate(start)
            print(f'DONE part {part_i + 1}/{n_chunks}; elapsed={(time.perf_counter() - t0) / 60:.1f} min; local remaining={start / 1024 ** 3:.2f} GB')

    copy_sidecars_to_dir(work_path, parts_dir)
    manifest['completed'] = True
    with chunk_manifest_path(work_path).open('w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
    if work_path.exists() and work_path.stat().st_size == 0:
        work_path.unlink()
    for suffix in sidecar_suffixes():
        sidecar = work_path.with_suffix(work_path.suffix + suffix)
        if sidecar.exists():
            sidecar.unlink()
    print(f'Chunked move complete for {work_path.name}: {parts_dir}')


def restore_chunked_from_results(work_path: Path) -> bool:
    if not RESTORE_LARGE_ARTIFACTS_FROM_RESULTS:
        return False
    manifest = load_chunk_manifest(work_path)
    if manifest is None:
        return False
    original_size = int(manifest['original_size'])
    n_chunks = int(manifest['n_chunks'])
    parts_dir = parts_dir_for_results(work_path)
    chunks = manifest_chunks(manifest)
    if work_path.exists() and work_path.stat().st_size == original_size:
        restore_sidecars_from_dir(work_path, parts_dir)
        return True
    if work_path.exists():
        print(f'Deleting incomplete local restore before retry: {work_path}')
        work_path.unlink()
    print(f'Reconstructing {work_path.name} from {n_chunks} Drive parts...')
    work_path.parent.mkdir(parents=True, exist_ok=True)
    buffer_size = int(CHUNKED_MOVE_BUFFER_MB * 1024 ** 2)
    with work_path.open('wb') as out:
        for part_i in range(n_chunks):
            part_key = f'{part_i:05d}'
            part_path = parts_dir / f'{work_path.name}.part{part_key}'
            expected_size = expected_chunk_size(manifest, part_i)
            expected_sha = chunks.get(part_key, {}).get('sha256')
            if not chunk_is_verified(part_path, expected_size, expected_sha, verify_checksum=VERIFY_DRIVE_CHUNK_CHECKSUMS):
                raise FileNotFoundError(f'Missing or incomplete chunk: {part_path}')
            print(f'RESTORE part {part_i + 1}/{n_chunks}: {expected_size / 1024 ** 3:.2f} GB')
            with part_path.open('rb') as f:
                while True:
                    block = f.read(buffer_size)
                    if not block:
                        break
                    out.write(block)
    if work_path.stat().st_size != original_size:
        raise OSError(f'Restored file size mismatch for {work_path.name}')
    restore_sidecars_from_dir(work_path, parts_dir)
    print(f'Restored {work_path.name} to scratch: {work_path}')
    return True


def move_legacy_artifact_if_needed(work_path: Path) -> None:
    if work_path.exists():
        return
    for legacy_dir in LEGACY_WORK_ARTIFACTS_DIRS:
        legacy_path = legacy_dir / work_path.name
        if not legacy_path.exists() or legacy_path.resolve() == work_path.resolve():
            continue
        work_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'Moving existing artifact from old work dir to scratch: {legacy_path} -> {work_path}')
        shutil.move(str(legacy_path), str(work_path))
        for suffix in ['.meta.json', '.timing.json']:
            legacy_sidecar = legacy_path.with_suffix(legacy_path.suffix + suffix)
            if legacy_sidecar.exists():
                shutil.move(str(legacy_sidecar), str(work_path.with_suffix(work_path.suffix + suffix)))
        return


def restore_from_results(work_path: Path) -> None:
    move_legacy_artifact_if_needed(work_path)
    if work_path.exists():
        if should_use_chunked_move(work_path):
            manifest = load_chunk_manifest(work_path)
            if manifest is not None and work_path.stat().st_size != int(manifest['original_size']):
                print(f'Deleting incomplete local artifact before chunk restore: {work_path}')
                work_path.unlink()
            else:
                return
        else:
            return
    if should_use_chunked_move(work_path) and restore_chunked_from_results(work_path):
        return
    result_path = path_for_results(work_path)
    if RESTORE_LARGE_ARTIFACTS_FROM_RESULTS and result_path.exists() and not work_path.exists():
        work_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'Restoring {result_path.name} to scratch...')
        shutil.copy2(result_path, work_path)
        for suffix in ['.meta.json', '.timing.json']:
            sidecar = result_path.with_suffix(result_path.suffix + suffix)
            if sidecar.exists():
                shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))


def copy_to_results(work_path: Path, delete_local_after_backup: bool = False) -> None:
    if not PERSIST_LARGE_ARTIFACTS_TO_DRIVE or not work_path.exists():
        return
    if should_use_chunked_move(work_path):
        if delete_local_after_backup:
            destructive_chunked_move_to_results(work_path)
        else:
            non_destructive_chunked_copy_to_results(work_path)
        return
    result_path = path_for_results(work_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists() and result_path.stat().st_size == work_path.stat().st_size:
        print(f'Drive backup already complete: {work_path.name}')
    else:
        if result_path.exists():
            print(f'Deleting incomplete Drive backup before retry: {result_path}')
            result_path.unlink()
        print(f'Backing up {work_path.name} to Drive results...')
        shutil.copy2(work_path, result_path)
        if result_path.stat().st_size != work_path.stat().st_size:
            raise OSError(f'Drive backup size mismatch for {work_path.name}; keeping local copy.')
    for suffix in ['.meta.json', '.timing.json']:
        sidecar = work_path.with_suffix(work_path.suffix + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, result_path.with_suffix(result_path.suffix + suffix))
    if delete_local_after_backup and DELETE_LOCAL_LARGE_ARTIFACTS_AFTER_DRIVE_BACKUP:
        if result_path.exists() and result_path.stat().st_size == work_path.stat().st_size:
            print(f'Deleting local scratch artifact after verified Drive backup: {work_path}')
            work_path.unlink()
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = work_path.with_suffix(work_path.suffix + suffix)
                if sidecar.exists():
                    sidecar.unlink()
        else:
            raise OSError(f'Backup verification failed for {work_path.name}; local copy was not deleted.')


def parquet_view(con: duckdb.DuckDBPyConnection, parquet_path: Path) -> None:
    path = str(parquet_path).replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW unique_surface_corpus AS SELECT * FROM read_parquet('{path}')")


def write_config() -> None:
    save_json(
        {
            'model_key': 'sapbert_contrastive',
            'model_dir': str(MODEL_DIR),
            'reference_csv_path': str(REFERENCE_CSV_PATH),
            'lab_labels_path': str(LAB_LABELS_PATH),
            'results_run_dir': str(RESULTS_RUN_DIR),
            'work_artifacts_dir': str(WORK_ARTIFACTS_DIR),
            'normalization': 'NFKC + lowercase + strip + whitespace collapse',
            'max_length': MAX_LENGTH,
            'pooling': POOLING,
            'embedding_dtype': EMBED_DTYPE,
            'cpu_threads': CPU_THREADS,
            'reference_chunksize': REFERENCE_CHUNKSIZE,
            'corpus_batch_size': CORPUS_BATCH_SIZE,
            'query_batch_size': QUERY_BATCH_SIZE,
            'index_type': INDEX_TYPE,
            'nlist': NLIST,
            'nprobe': NPROBE,
            'train_size': TRAIN_SIZE,
            'add_batch_size': ADD_BATCH_SIZE,
            'search_batch_size': SEARCH_BATCH_SIZE,
            'rerank_candidates_with_original_embeddings': RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS,
            'rerank_batch_size': RERANK_BATCH_SIZE,
            'persist_large_artifacts_to_drive': PERSIST_LARGE_ARTIFACTS_TO_DRIVE,
            'delete_local_large_artifacts_after_drive_backup': DELETE_LOCAL_LARGE_ARTIFACTS_AFTER_DRIVE_BACKUP,
            'copy_large_artifacts_to_results': COPY_LARGE_ARTIFACTS_TO_RESULTS,
            'chunked_move_artifact_names': sorted(CHUNKED_MOVE_ARTIFACT_NAMES),
            'chunked_move_chunk_gb': CHUNKED_MOVE_CHUNK_GB,
            'chunked_move_buffer_mb': CHUNKED_MOVE_BUFFER_MB,
            'chunked_move_write_checksums': CHUNKED_MOVE_WRITE_CHECKSUMS,
            'verify_drive_chunk_checksums': VERIFY_DRIVE_CHUNK_CHECKSUMS,
            'delete_prepare_db_after_parquet': DELETE_PREPARE_DB_AFTER_PARQUET,
            'allow_cpu_ivf_flat_fallback': ALLOW_CPU_IVF_FLAT_FALLBACK,
            'm_pq': M_PQ,
            'nbits': NBITS,
            'k_retrieve': K_RETRIEVE,
            'k_values': K_VALUES,
            'filter_identical_normalized_surface_forms': True,
            'evaluate_only_queries_with_nonexact_relevant': EVALUATE_ONLY_QUERIES_WITH_NONEXACT_RELEVANT,
        },
        RESULTS_RUN_DIR / 'config.json',
    )


write_config()


## Prepare Deduplicated Reference Corpus

This stage streams the 114M-row reference CSV, normalizes names, drops duplicate `(normalized_name, cid)` pairs within each chunk, stores them in DuckDB on local scratch, then writes one row per unique normalized surface form to Parquet. Candidate CID ambiguity is preserved in `candidate_cids`.

In [ ]:
def prepare_unique_surface_corpus() -> dict:
    restore_from_results(CORPUS_PARQUET_PATH)
    if CORPUS_PARQUET_PATH.exists() and not REBUILD_PREPARED_CORPUS:
        pf = pq.ParquetFile(CORPUS_PARQUET_PATH)
        return {'loaded_from_cache': True, 'unique_surface_rows': int(pf.metadata.num_rows)}

    if REBUILD_PREPARED_CORPUS:
        for path in [PREPARE_DB_PATH, CORPUS_PARQUET_PATH]:
            if path.exists():
                path.unlink()

    con = duckdb.connect(str(PREPARE_DB_PATH))
    con.execute(f'PRAGMA threads={CPU_THREADS}')
    con.execute('DROP TABLE IF EXISTS reference_pairs')
    con.execute('CREATE TABLE reference_pairs(normalized_name VARCHAR, cid VARCHAR, display_name VARCHAR)')

    total_rows = 0
    inserted_pairs = 0
    t0 = time.perf_counter()
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype='string',
        chunksize=REFERENCE_CHUNKSIZE,
        on_bad_lines='skip',
    )

    for chunk_i, chunk in enumerate(reader, start=1):
        total_rows += len(chunk)
        chunk = chunk.dropna(subset=[CID_COL, NAME_COL]).copy()
        chunk['normalized_name'] = normalize_series(chunk[NAME_COL])
        chunk = chunk[chunk['normalized_name'].str.len() > 0]
        chunk = chunk.rename(columns={CID_COL: 'cid', NAME_COL: 'display_name'})
        chunk = chunk[['normalized_name', 'cid', 'display_name']].drop_duplicates(['normalized_name', 'cid'])
        inserted_pairs += len(chunk)
        con.register('chunk_df', chunk)
        con.execute('INSERT INTO reference_pairs SELECT normalized_name, CAST(cid AS VARCHAR), display_name FROM chunk_df')
        con.unregister('chunk_df')

        if chunk_i == 1 or chunk_i % 5 == 0:
            elapsed_min = (time.perf_counter() - t0) / 60
            print(f'chunk={chunk_i:,}; source rows={total_rows:,}; inserted chunk pairs={inserted_pairs:,}; elapsed={elapsed_min:.1f} min')

    print('Building one-row-per-normalized-surface corpus in DuckDB...')
    con.execute('DROP TABLE IF EXISTS unique_surface_corpus')
    con.execute(
        '''
        CREATE TABLE unique_surface_corpus AS
        WITH distinct_pairs AS (
            SELECT normalized_name, cid, MIN(display_name) AS display_name
            FROM reference_pairs
            GROUP BY normalized_name, cid
        ), grouped AS (
            SELECT
                normalized_name,
                ANY_VALUE(display_name) AS display_name,
                STRING_AGG(cid, '|' ORDER BY cid) AS candidate_cids,
                COUNT(*) AS n_candidate_cids
            FROM distinct_pairs
            GROUP BY normalized_name
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY normalized_name) - 1 AS surface_id,
            normalized_name,
            display_name,
            candidate_cids,
            n_candidate_cids
        FROM grouped
        ORDER BY surface_id
        '''
    )
    parquet_path = str(CORPUS_PARQUET_PATH).replace("'", "''")
    con.execute(f"COPY unique_surface_corpus TO '{parquet_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    summary = con.execute(
        '''
        SELECT
            COUNT(*) AS unique_surface_rows,
            SUM(CASE WHEN n_candidate_cids > 1 THEN 1 ELSE 0 END) AS ambiguous_surface_rows,
            MAX(n_candidate_cids) AS max_candidate_cids
        FROM unique_surface_corpus
        '''
    ).fetchdf().iloc[0].to_dict()
    con.close()
    if DELETE_PREPARE_DB_AFTER_PARQUET and PREPARE_DB_PATH.exists():
        print('Deleting temporary DuckDB prepare database to free scratch space...')
        PREPARE_DB_PATH.unlink()
    summary = {k: int(v) for k, v in summary.items()}
    summary.update({'loaded_from_cache': False, 'reference_rows_read': int(total_rows), 'inserted_chunk_pairs': int(inserted_pairs)})
    copy_to_results(CORPUS_PARQUET_PATH, delete_local_after_backup=True)
    return summary


corpus_summary = prepare_unique_surface_corpus()
save_json(corpus_summary, RESULTS_RUN_DIR / 'corpus_summary.json')
display(pd.DataFrame([corpus_summary]).T.rename(columns={0: 'value'}))


## Encode Unique Reference Surface Forms

In [ ]:
def encode_parquet_texts_to_memmap(
    parquet_path: Path,
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    max_length: int,
    device: str,
    pooling: str,
    normalize: bool,
    out_dtype: str,
    force_reencode: bool = False,
) -> np.ndarray:
    import torch

    restore_from_results(out_path)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    pf = pq.ParquetFile(parquet_path)
    n = int(pf.metadata.num_rows)
    d = int(model.config.hidden_size)
    expected_meta = {
        'n': n,
        'd': d,
        'source_parquet': str(parquet_path),
        'model_name': model_name,
        'pooling': pooling,
        'max_length': int(max_length),
        'dtype': out_dtype,
        'normalize': bool(normalize),
    }

    if out_path.exists() and meta_path.exists() and not force_reencode:
        try:
            with meta_path.open('r', encoding='utf-8') as f:
                old_meta = json.load(f)
            arr = np.load(out_path, mmap_mode='r')
            stable_keys = ['n', 'd', 'model_name', 'pooling', 'max_length', 'dtype', 'normalize']
            old_stable = {key: old_meta.get(key) for key in stable_keys}
            expected_stable = {key: expected_meta.get(key) for key in stable_keys}
            if old_stable == expected_stable and arr.shape == (n, d):
                print('Using cached corpus embeddings:', out_path)
                return arr
        except Exception:
            pass

    if out_path.exists():
        out_path.unlink()
    for stale_sidecar in [meta_path, out_path.with_suffix(out_path.suffix + '.timing.json')]:
        if stale_sidecar.exists():
            stale_sidecar.unlink()
    arr = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.dtype(out_dtype), shape=(n, d))
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    offset = 0
    t0 = time.perf_counter()

    for batch in tqdm(pf.iter_batches(columns=['normalized_name'], batch_size=batch_size), total=(n + batch_size - 1) // batch_size, desc='Encoding unique reference surfaces'):
        texts = batch.column(0).to_pylist()
        enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
        with torch.inference_mode():
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], pooling)
            if normalize:
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        end = offset + len(texts)
        arr[offset:end] = emb.detach().cpu().numpy().astype(np.dtype(out_dtype), copy=False)
        offset = end
        del texts, enc, out, emb
        if device.startswith('cuda'):
            torch.cuda.empty_cache()

    arr.flush()
    elapsed_sec = float(time.perf_counter() - t0)
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': elapsed_sec}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


restore_from_results(CORPUS_PARQUET_PATH)
device = choose_device('auto')
print('Embedding device:', device)
tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)
corpus_embeddings = encode_parquet_texts_to_memmap(
    parquet_path=CORPUS_PARQUET_PATH,
    out_path=CORPUS_EMBEDDINGS_PATH,
    tokenizer=tokenizer,
    model=hf_model,
    model_name=str(MODEL_DIR),
    batch_size=CORPUS_BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=device,
    pooling=POOLING,
    normalize=L2_NORMALIZE,
    out_dtype=EMBED_DTYPE,
    force_reencode=REENCODE_CORPUS,
)
print('Corpus embeddings:', corpus_embeddings.shape, corpus_embeddings.dtype)

# Store expensive corpus artifacts in Drive and free scratch before the next stage.
import gc
del corpus_embeddings
gc.collect()
# Keep the local corpus embeddings for the immediately following FAISS build.
# They are deleted later after FAISS/reranking no longer needs them.
copy_to_results(CORPUS_EMBEDDINGS_PATH, delete_local_after_backup=False)
copy_to_results(CORPUS_PARQUET_PATH, delete_local_after_backup=True)


## Encode External Lab Queries

In [ ]:
import gc

lab_queries = pd.read_csv(LAB_LABELS_PATH, dtype='string')
lab_queries = lab_queries.dropna(subset=['cid', 'name']).copy()
lab_queries['cid'] = lab_queries['cid'].astype(str)
lab_queries['normalized_name'] = normalize_series(lab_queries['name'])
lab_queries = lab_queries[lab_queries['normalized_name'].str.len() > 0].reset_index(drop=True)
lab_queries['query_i'] = np.arange(len(lab_queries), dtype=np.int64)
lab_queries[['query_i', 'cid', 'name', 'normalized_name']].to_csv(RESULTS_ARTIFACTS_DIR / 'external_lab_queries.csv', index=False)

restore_from_results(QUERY_EMBEDDINGS_PATH)
if REENCODE_QUERIES and QUERY_EMBEDDINGS_PATH.exists():
    QUERY_EMBEDDINGS_PATH.unlink()
query_embeddings = encode_texts_to_memmap(
    texts=lab_queries['normalized_name'].astype(str).tolist(),
    out_path=QUERY_EMBEDDINGS_PATH,
    tokenizer=tokenizer,
    model=hf_model,
    model_name=str(MODEL_DIR),
    prefix=None,
    pooling=POOLING,
    batch_size=QUERY_BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=device,
    normalize=L2_NORMALIZE,
    out_dtype=EMBED_DTYPE,
    run_key='external_lab_sapbert_contrastive',
)
print('Lab queries:', len(lab_queries))
print('Query embeddings:', query_embeddings.shape, query_embeddings.dtype)
del query_embeddings
gc.collect()
copy_to_results(QUERY_EMBEDDINGS_PATH, delete_local_after_backup=True)


## Build FAISS Index

In [ ]:
import gc

if device.startswith('cuda'):
    print('Releasing transformer model GPU memory before FAISS...')
    del hf_model
    del tokenizer
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception:
        pass

def restore_sidecars_only(work_path: Path) -> None:
    if should_use_chunked_move(work_path):
        restore_sidecars_from_dir(work_path, parts_dir_for_results(work_path))
        return
    result_path = path_for_results(work_path)
    for suffix in sidecar_suffixes():
        sidecar = result_path.with_suffix(result_path.suffix + suffix)
        if sidecar.exists():
            work_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))


def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def expected_faiss_meta(n: int, d: int) -> dict:
    if INDEX_TYPE != 'flat':
        nlist_eff = min(max(int(NLIST), 1), max(int(n), 1), max(int(TRAIN_SIZE), 1))
    else:
        nlist_eff = int(NLIST)
    return {
        'index_type': INDEX_TYPE,
        'n': int(n),
        'd': int(d),
        'nlist': int(nlist_eff),
        'train_size': int(TRAIN_SIZE),
        'm_pq': int(M_PQ),
        'nbits': int(NBITS),
        'seed': int(SEED),
        'run_key': 'external_lab_unique_surface_sapbert_contrastive',
    }


def move_faiss_index_to_device(cpu_index, faiss_device: str):
    import faiss

    if faiss_gpu_available(faiss, faiss_device):
        gpu_id = int(faiss_device.split(':', 1)[1]) if ':' in faiss_device else 0
        res = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = bool(USE_FLOAT16_FAISS)
        index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index, opts)
    else:
        index = cpu_index
    if hasattr(index, 'nprobe'):
        index.nprobe = int(NPROBE)
    return index


def load_cached_faiss_index(expected_meta: dict, faiss_device: str):
    import faiss

    restore_from_results(FAISS_INDEX_PATH)
    meta_path = FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json')
    old_meta = load_json_if_exists(meta_path)
    if not FAISS_INDEX_PATH.exists() or old_meta != expected_meta:
        return None
    print('Using cached FAISS index without restoring full corpus embeddings:', FAISS_INDEX_PATH)
    cpu_index = faiss.read_index(str(FAISS_INDEX_PATH))
    return move_faiss_index_to_device(cpu_index, faiss_device)


restore_sidecars_only(CORPUS_EMBEDDINGS_PATH)
corpus_embedding_meta = load_json_if_exists(CORPUS_EMBEDDINGS_PATH.with_suffix(CORPUS_EMBEDDINGS_PATH.suffix + '.meta.json'))
corpus_embeddings = None
if corpus_embedding_meta is not None:
    n_corpus = int(corpus_embedding_meta['n'])
    dim = int(corpus_embedding_meta['d'])
else:
    restore_from_results(CORPUS_EMBEDDINGS_PATH)
    corpus_embeddings = np.load(CORPUS_EMBEDDINGS_PATH, mmap_mode='r')
    n_corpus, dim = corpus_embeddings.shape
gpu_ivf_flat_gb = n_corpus * dim * (2 if USE_FLOAT16_FAISS else 4) / (1024 ** 3)
cpu_ivf_flat_gb = n_corpus * dim * 4 / (1024 ** 3)
ivfpq_codes_gb = n_corpus * M_PQ / (1024 ** 3)
if INDEX_TYPE == 'ivf_flat':
    estimated_index_gb = gpu_ivf_flat_gb
elif INDEX_TYPE == 'ivfpq':
    estimated_index_gb = ivfpq_codes_gb
else:
    estimated_index_gb = gpu_ivf_flat_gb
faiss_device = 'cuda' if USE_GPU_FAISS_IF_FITS and estimated_index_gb <= GPU_INDEX_MEMORY_LIMIT_GB else 'cpu'
if INDEX_TYPE == 'ivf_flat' and faiss_device == 'cpu' and not ALLOW_CPU_IVF_FLAT_FALLBACK:
    raise RuntimeError(
        f'Full IVF-flat does not fit the GPU estimate ({gpu_ivf_flat_gb:.2f} GB > {GPU_INDEX_MEMORY_LIMIT_GB:.2f} GB), '
        f'and a CPU IVF-flat index would be roughly {cpu_ivf_flat_gb:.2f} GB before overhead. '
        "Set INDEX_TYPE = 'ivfpq' for a compressed index, reduce the corpus, or use a larger GPU/disk."
    )
print(f'Estimated GPU IVF-flat vector storage: {gpu_ivf_flat_gb:.2f} GB')
print(f'Estimated CPU IVF-flat vector storage: {cpu_ivf_flat_gb:.2f} GB')
print(f'Estimated IVFPQ code storage with M_PQ={M_PQ}: {ivfpq_codes_gb:.2f} GB')
print('FAISS device:', faiss_device)

if REBUILD_FAISS_INDEX:
    for stale_path in [
        FAISS_INDEX_PATH,
        FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json'),
        FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.timing.json'),
    ]:
        if stale_path.exists():
            stale_path.unlink()

faiss_expected_meta = expected_faiss_meta(n_corpus, dim)
faiss_index = None if REBUILD_FAISS_INDEX else load_cached_faiss_index(faiss_expected_meta, faiss_device)
if faiss_index is None:
    if corpus_embeddings is None:
        restore_from_results(CORPUS_EMBEDDINGS_PATH)
        corpus_embeddings = np.load(CORPUS_EMBEDDINGS_PATH, mmap_mode='r')
    faiss_index, _ = build_faiss_index_from_memmap(
        xb_memmap=corpus_embeddings,
        index_path=FAISS_INDEX_PATH,
        index_type=INDEX_TYPE,
        nlist=NLIST,
        nprobe=NPROBE,
        train_size=TRAIN_SIZE,
        add_batch_size=ADD_BATCH_SIZE,
        m_pq=M_PQ,
        nbits=NBITS,
        seed=SEED,
        device=faiss_device,
        use_float16=USE_FLOAT16_FAISS,
        run_key='external_lab_unique_surface_sapbert_contrastive',
    )
    del corpus_embeddings
    gc.collect()
    # Keep local embeddings for the search/evaluation cell, which reranks IVFPQ candidates with original vectors.
    copy_to_results(CORPUS_EMBEDDINGS_PATH, delete_local_after_backup=False)
elif CORPUS_EMBEDDINGS_PATH.exists():
    # If the embedding file is already local, make sure the Drive checkpoint is still present.
    copy_to_results(CORPUS_EMBEDDINGS_PATH, delete_local_after_backup=False)
copy_to_results(FAISS_INDEX_PATH, delete_local_after_backup=True)


## Search and Evaluate with Exact Surface Filtering

In [ ]:
def compute_available_relevant_counts(con: duckdb.DuckDBPyConnection, queries: pd.DataFrame) -> pd.DataFrame:
    query_cids = pd.DataFrame({'cid': sorted(queries['cid'].astype(str).unique())})
    con.register('query_cids', query_cids)
    cid_counts = con.execute(
        '''
        WITH exploded AS (
            SELECT UNNEST(STRING_SPLIT(candidate_cids, '|')) AS cid
            FROM unique_surface_corpus
        )
        SELECT e.cid, COUNT(*) AS corpus_surface_count
        FROM exploded e
        JOIN query_cids q ON e.cid = q.cid
        GROUP BY e.cid
        '''
    ).fetchdf()
    con.unregister('query_cids')
    count_by_cid = dict(zip(cid_counts['cid'].astype(str), cid_counts['corpus_surface_count'].astype(int)))

    query_meta = queries[['query_i', 'cid', 'normalized_name']].copy()
    con.register('query_meta', query_meta)
    exact_counts = con.execute(
        '''
        SELECT q.query_i, COUNT(*) AS exact_surface_count
        FROM query_meta q
        JOIN unique_surface_corpus u ON u.normalized_name = q.normalized_name
        WHERE LIST_CONTAINS(STRING_SPLIT(u.candidate_cids, '|'), q.cid)
        GROUP BY q.query_i
        '''
    ).fetchdf()
    con.unregister('query_meta')
    exact_by_query = dict(zip(exact_counts['query_i'].astype(int), exact_counts['exact_surface_count'].astype(int)))

    out = queries[['query_i', 'cid']].copy()
    out['corpus_surface_count_for_cid'] = out['cid'].map(count_by_cid).fillna(0).astype(int)
    out['exact_surface_count'] = out['query_i'].map(exact_by_query).fillna(0).astype(int)
    out['available_nonexact_relevant'] = (out['corpus_surface_count_for_cid'] - out['exact_surface_count']).clip(lower=0).astype(int)
    return out


def make_neighbor_long(neighbor_indices: np.ndarray) -> pd.DataFrame:
    n_queries, k = neighbor_indices.shape
    return pd.DataFrame(
        {
            'query_i': np.repeat(np.arange(n_queries, dtype=np.int64), k),
            'rank0': np.tile(np.arange(k, dtype=np.int16), n_queries),
            'surface_id': neighbor_indices.reshape(-1).astype(np.int64, copy=False),
        }
    )


def compute_metrics(hit_df: pd.DataFrame, eval_queries: pd.DataFrame, k_values: Iterable[int]) -> dict:
    n = max(len(eval_queries), 1)
    relevant_hits = hit_df[hit_df['relevant'].astype(bool)]
    first_rel = relevant_hits.groupby('query_i')['filtered_rank'].min()
    metrics = {'n_queries': int(len(eval_queries)), 'MRR': float((1.0 / first_rel).sum() / n)}
    available = eval_queries.set_index('query_i')['available_nonexact_relevant'].astype(float)
    for k in k_values:
        topk = hit_df[hit_df['filtered_rank'] <= int(k)]
        rel_at_k = topk.groupby('query_i')['relevant'].sum().reindex(eval_queries['query_i'], fill_value=0).astype(float)
        denom = available.reindex(eval_queries['query_i']).clip(lower=1.0)
        metrics[f'HitRate@{int(k)}'] = float((rel_at_k > 0).sum() / n)
        metrics[f'Precision@{int(k)}'] = float((rel_at_k / int(k)).sum() / n)
        metrics[f'RecallFrac@{int(k)}'] = float((rel_at_k / denom).sum() / n)
    return metrics


def rerank_neighbors_with_original_embeddings(
    query_embeddings: np.ndarray,
    corpus_embeddings: np.ndarray,
    neighbor_indices: np.ndarray,
    batch_size: int,
) -> np.ndarray:
    reranked = np.empty_like(neighbor_indices)
    n_queries = len(neighbor_indices)
    for start in tqdm(range(0, n_queries, int(batch_size)), desc='Reranking FAISS candidates with original embeddings'):
        end = min(start + int(batch_size), n_queries)
        q = np.asarray(query_embeddings[start:end], dtype=np.float32)
        candidates = np.asarray(neighbor_indices[start:end], dtype=np.int64)
        valid = candidates >= 0
        safe_candidates = np.where(valid, candidates, 0)
        xb = np.asarray(corpus_embeddings[safe_candidates.reshape(-1)], dtype=np.float32).reshape(
            candidates.shape[0], candidates.shape[1], -1
        )
        scores = np.einsum('bd,bkd->bk', q, xb, optimize=True)
        scores[~valid] = -np.inf
        order = np.argsort(-scores, axis=1)
        reranked[start:end] = np.take_along_axis(candidates, order, axis=1)
        del q, candidates, valid, safe_candidates, xb, scores, order
    return reranked


def neighbor_cache_config() -> dict:
    return {
        'model_key': 'sapbert_contrastive',
        'model_dir': str(MODEL_DIR),
        'lab_labels': file_fingerprint(LAB_LABELS_PATH),
        'query_embeddings': artifact_fingerprint(QUERY_EMBEDDINGS_PATH),
        'corpus_embeddings_for_rerank': artifact_fingerprint(CORPUS_EMBEDDINGS_PATH) if RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS else None,
        'faiss_index': artifact_fingerprint(FAISS_INDEX_PATH),
        'index_type': INDEX_TYPE,
        'nlist': NLIST,
        'nprobe': NPROBE,
        'train_size': TRAIN_SIZE,
        'm_pq': M_PQ,
        'nbits': NBITS,
        'k_retrieve': K_RETRIEVE,
        'search_batch_size': SEARCH_BATCH_SIZE,
        'rerank_candidates_with_original_embeddings': RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS,
        'rerank_batch_size': RERANK_BATCH_SIZE,
        'pooling': POOLING,
        'max_length': MAX_LENGTH,
        'embedding_dtype': EMBED_DTYPE,
        'l2_normalize': L2_NORMALIZE,
    }


def load_cached_neighbors(expected_config: dict, n_queries: int) -> np.ndarray | None:
    restore_from_results(NEIGHBOR_CONFIG_PATH)
    restore_from_results(NEIGHBOR_INDICES_PATH)
    if not NEIGHBOR_INDICES_PATH.exists() or not NEIGHBOR_CONFIG_PATH.exists():
        return None
    try:
        with NEIGHBOR_CONFIG_PATH.open('r', encoding='utf-8') as f:
            old_config = json.load(f)
        arr = np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r')
        if old_config == expected_config and arr.shape == (int(n_queries), int(K_RETRIEVE)):
            print('Using validated cached neighbor indices:', NEIGHBOR_INDICES_PATH)
            return arr
        print('Ignoring stale neighbor cache; config or shape changed.')
    except Exception as exc:
        print('Ignoring unreadable neighbor cache:', exc)
    for stale_path in [NEIGHBOR_INDICES_PATH, NEIGHBOR_CONFIG_PATH]:
        if stale_path.exists():
            stale_path.unlink()
    return None


def save_neighbor_cache(neighbor_indices: np.ndarray, config: dict) -> None:
    np.save(NEIGHBOR_INDICES_PATH, neighbor_indices)
    save_json(config, NEIGHBOR_CONFIG_PATH)
    copy_to_results(NEIGHBOR_INDICES_PATH, delete_local_after_backup=True)
    copy_to_results(NEIGHBOR_CONFIG_PATH, delete_local_after_backup=False)


restore_from_results(QUERY_EMBEDDINGS_PATH)
query_embeddings = np.load(QUERY_EMBEDDINGS_PATH, mmap_mode='r')
expected_neighbor_config = neighbor_cache_config()
neighbor_indices = load_cached_neighbors(expected_neighbor_config, n_queries=len(query_embeddings))
if neighbor_indices is not None:
    search_elapsed = None
else:
    neighbor_indices, search_elapsed = search_faiss_neighbors(
        index=faiss_index,
        query_embeddings=query_embeddings,
        k_retrieve=K_RETRIEVE,
        batch_size=SEARCH_BATCH_SIZE,
    )
    if RERANK_CANDIDATES_WITH_ORIGINAL_EMBEDDINGS:
        restore_from_results(CORPUS_EMBEDDINGS_PATH)
        corpus_embeddings_for_rerank = np.load(CORPUS_EMBEDDINGS_PATH, mmap_mode='r')
        neighbor_indices = rerank_neighbors_with_original_embeddings(
            query_embeddings=query_embeddings,
            corpus_embeddings=corpus_embeddings_for_rerank,
            neighbor_indices=neighbor_indices,
            batch_size=RERANK_BATCH_SIZE,
        )
        del corpus_embeddings_for_rerank
        gc.collect()
        copy_to_results(CORPUS_EMBEDDINGS_PATH, delete_local_after_backup=True)
    save_neighbor_cache(neighbor_indices, expected_neighbor_config)

con_eval = duckdb.connect()
restore_from_results(CORPUS_PARQUET_PATH)
parquet_view(con_eval, CORPUS_PARQUET_PATH)
availability = compute_available_relevant_counts(con_eval, lab_queries)
eval_queries = lab_queries.merge(availability[['query_i', 'available_nonexact_relevant']], on='query_i', how='left')
if EVALUATE_ONLY_QUERIES_WITH_NONEXACT_RELEVANT:
    eval_queries = eval_queries[eval_queries['available_nonexact_relevant'] > 0].copy()
eval_query_ids = set(eval_queries['query_i'].astype(int))
print('All labeled queries:', len(lab_queries))
print('Evaluated queries with non-exact relevant surfaces:', len(eval_queries))

neighbor_long = make_neighbor_long(np.asarray(neighbor_indices))
neighbor_long = neighbor_long[neighbor_long['query_i'].isin(eval_query_ids) & (neighbor_long['surface_id'] >= 0)].copy()
query_meta = eval_queries[['query_i', 'cid', 'normalized_name', 'name']].copy()
con_eval.register('neighbors', neighbor_long)
con_eval.register('query_meta', query_meta)
hit_df = con_eval.execute(
    f'''
    WITH filtered AS (
        SELECT
            n.query_i,
            n.rank0,
            n.surface_id,
            q.cid AS query_cid,
            q.name AS query_name,
            u.normalized_name AS retrieved_normalized_name,
            u.display_name AS retrieved_display_name,
            u.candidate_cids,
            LIST_CONTAINS(STRING_SPLIT(u.candidate_cids, '|'), q.cid) AS relevant,
            ROW_NUMBER() OVER (PARTITION BY n.query_i ORDER BY n.rank0) AS filtered_rank
        FROM neighbors n
        JOIN query_meta q ON n.query_i = q.query_i
        JOIN unique_surface_corpus u ON n.surface_id = u.surface_id
        WHERE u.normalized_name <> q.normalized_name
    )
    SELECT *
    FROM filtered
    WHERE filtered_rank <= {max(K_VALUES)}
    ORDER BY query_i, filtered_rank
    '''
).fetchdf()
con_eval.unregister('neighbors')
con_eval.unregister('query_meta')

metrics = compute_metrics(hit_df, eval_queries, K_VALUES)
payload = {
    'run_name': 'sapbert_contrastive_external_lab_unique_surface',
    'model_key': 'sapbert_contrastive',
    'model_dir': str(MODEL_DIR),
    'metrics': metrics,
    'search_elapsed_sec': search_elapsed,
    'n_all_labeled_queries': int(len(lab_queries)),
    'n_evaluated_queries': int(len(eval_queries)),
    'n_queries_excluded_no_nonexact_relevant': int(len(lab_queries) - len(eval_queries)),
    'corpus_summary': corpus_summary,
    'exact_surface_form_hits_filtered': True,
}
save_json(payload, RESULTS_RUN_DIR / 'external_lab_metrics.json')
availability.to_csv(RESULTS_ARTIFACTS_DIR / 'query_available_relevant_counts.csv', index=False)
hit_df.to_csv(RESULTS_RUN_DIR / 'external_lab_top_hits.csv', index=False)

def split_candidate_cids(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split('|') if part.strip()]


top_predictions = hit_df[hit_df['filtered_rank'] == 1].copy()
top_predictions['proposed_candidate_cids'] = top_predictions['candidate_cids'].fillna('').astype(str)
top_predictions['proposed_candidate_cid_list'] = top_predictions['proposed_candidate_cids'].map(split_candidate_cids)
top_predictions['proposed_n_candidate_cids'] = top_predictions['proposed_candidate_cid_list'].map(len).astype(int)
top_predictions['proposed_cid_is_ambiguous'] = top_predictions['proposed_n_candidate_cids'] > 1
top_predictions['proposed_cid'] = top_predictions['proposed_candidate_cid_list'].map(
    lambda ids: ids[0] if len(ids) == 1 else ''
)
top_predictions['gold_cid_in_proposed_candidates'] = top_predictions.apply(
    lambda row: str(row['query_cid']).strip() in set(row['proposed_candidate_cid_list']),
    axis=1,
)
top_predictions_for_join = top_predictions.rename(
    columns={
        'query_name': 'lab_name',
        'query_cid': 'gold_cid',
        'retrieved_display_name': 'top_retrieved_name',
    }
)[[
    'query_i',
    'lab_name',
    'gold_cid',
    'proposed_cid',
    'proposed_candidate_cids',
    'proposed_n_candidate_cids',
    'proposed_cid_is_ambiguous',
    'gold_cid_in_proposed_candidates',
    'top_retrieved_name',
    'retrieved_normalized_name',
    'surface_id',
    'filtered_rank',
]]
base_predictions = eval_queries[[
    'query_i',
    'name',
    'cid',
    'normalized_name',
    'available_nonexact_relevant',
]].rename(
    columns={
        'name': 'lab_name',
        'cid': 'gold_cid',
        'normalized_name': 'lab_normalized_name',
    }
)
model_cid_predictions = base_predictions.merge(
    top_predictions_for_join,
    on=['query_i', 'lab_name', 'gold_cid'],
    how='left',
)
model_cid_predictions['prediction_status'] = np.where(
    model_cid_predictions['top_retrieved_name'].notna(),
    'predicted',
    'no_nonexact_hit_in_retrieved_pool',
)
for col in ['proposed_cid', 'proposed_candidate_cids', 'top_retrieved_name', 'retrieved_normalized_name', 'surface_id']:
    model_cid_predictions[col] = model_cid_predictions[col].fillna('')
model_cid_predictions['proposed_n_candidate_cids'] = model_cid_predictions['proposed_n_candidate_cids'].fillna(0).astype(int)
model_cid_predictions['proposed_cid_is_ambiguous'] = model_cid_predictions['proposed_cid_is_ambiguous'].fillna(False).astype(bool)
model_cid_predictions['gold_cid_in_proposed_candidates'] = model_cid_predictions['gold_cid_in_proposed_candidates'].fillna(False).astype(bool)
model_cid_predictions['filtered_rank'] = model_cid_predictions['filtered_rank'].astype('Int64')
prediction_cols = [
    'query_i',
    'lab_name',
    'gold_cid',
    'lab_normalized_name',
    'available_nonexact_relevant',
    'prediction_status',
    'proposed_cid',
    'proposed_candidate_cids',
    'proposed_n_candidate_cids',
    'proposed_cid_is_ambiguous',
    'gold_cid_in_proposed_candidates',
    'top_retrieved_name',
    'retrieved_normalized_name',
    'surface_id',
    'filtered_rank',
]
model_cid_predictions = model_cid_predictions[prediction_cols]
model_cid_predictions.to_csv(RESULTS_RUN_DIR / 'external_lab_model_cid_predictions.csv', index=False)
print(pretty_metrics(metrics, K_VALUES))


## Save Examples

In [ ]:
rng = np.random.default_rng(SEED)
relevant_queries = set(hit_df.loc[hit_df['relevant'].astype(bool), 'query_i'].astype(int))
eval_queries_for_examples = eval_queries.copy()
eval_queries_for_examples['has_hit_in_top_k'] = eval_queries_for_examples['query_i'].astype(int).isin(relevant_queries)
sample_query_ids = []
for flag in [True, False]:
    pool = eval_queries_for_examples[eval_queries_for_examples['has_hit_in_top_k'] == flag]['query_i'].to_numpy(dtype=np.int64)
    if len(pool):
        sample_query_ids.extend(rng.choice(pool, size=min(25, len(pool)), replace=False).tolist())

examples = hit_df[hit_df['query_i'].isin(sample_query_ids)].copy()
examples.to_csv(RESULTS_RUN_DIR / 'external_lab_examples.csv', index=False)

summary = pd.DataFrame(
    [
        {'field': 'result_dir', 'value': str(RESULTS_RUN_DIR)},
        {'field': 'unique_surface_corpus', 'value': str(artifact_storage_path(CORPUS_PARQUET_PATH))},
        {'field': 'corpus_embeddings', 'value': str(artifact_storage_path(CORPUS_EMBEDDINGS_PATH))},
        {'field': 'query_embeddings', 'value': str(artifact_storage_path(QUERY_EMBEDDINGS_PATH))},
        {'field': 'faiss_index', 'value': str(artifact_storage_path(FAISS_INDEX_PATH))},
        {'field': 'metrics', 'value': str(RESULTS_RUN_DIR / 'external_lab_metrics.json')},
        {'field': 'model_cid_predictions', 'value': str(RESULTS_RUN_DIR / 'external_lab_model_cid_predictions.csv')},
        {'field': 'top_hits', 'value': str(RESULTS_RUN_DIR / 'external_lab_top_hits.csv')},
    ]
)
summary.to_csv(RESULTS_RUN_DIR / 'artifact_manifest.csv', index=False)
display(summary)
display(pd.DataFrame([metrics]).T.rename(columns={0: 'value'}))
display(model_cid_predictions.head(25))
